# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0mneeha93/ML-Track/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded, length:", len(hf_token))
!pip install -q duckdb huggingface_hub

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB ready.")

Token loaded, length: 37
DuckDB ready.
Token loaded, length: 37
DuckDB ready.


## 2. Feature notes (meaning, missing, categorical, available-when?)

**Feature fields** (used for clustering): `word_count`, `impressions_90d`-equivalent daily signals, `ctr`, `avg_position` (called `gsc_avg_position` in the warehouse), `content_age_days`, `days_since_last_update`, engagement/scroll signals.

**Label fields:** none clustering is unsupervised, no target label.

**Context fields** (not used as features, only for grouping/joining/inspection): `content_hash_id`, `client_hash_id`, `report_date`.

**Excluded, with why:** `trend_direction` and `trend_pct` excluded because they're derived/proxy fields tied to a decision outcome, not raw observable behavior; including them risks circularity even in an unsupervised setting, since I don't want cluster structure to trivially mirror an existing bucket. Also excluding any FlyRank product decision flags (`health_score`, `priority_score`) these aren't shipped in the warehouse anyway, so there's nothing to accidentally include.

In [2]:
grain_check = con.execute("""
    SELECT report_date, content_hash_id, COUNT(*) as cnt
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows with duplicate grain (should be empty):")
print(grain_check)
print("Empty = grain confirmed: one row per page per day.")
features_df = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        SUM(f.gsc_clicks) as clicks_total,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions)
             ELSE NULL END as ctr,
        c.word_count,
        DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') as days_since_last_update
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, c.word_count, c.content_updated_date
    LIMIT 1000
""").df()
print(features_df.shape)
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with duplicate grain (should be empty):
Empty DataFrame
Columns: [report_date, content_hash_id, cnt]
Index: []
Empty = grain confirmed: one row per page per day.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1000, 7)


,content_hash_id,avg_position,impressions_total,clicks_total,ctr,word_count,days_since_last_update
0,content_2e6360ad20fd7107,5.145765,899.0,1.0,0.001112,2855,-90
1,content_65c50dfe9d87a585,6.969536,3108.0,0.0,0.000000,2779,-90
2,content_cec711b02f3bbde6,4.428747,602.0,4.0,0.006645,2455,-90
3,content_614baf2af4330bd7,4.685335,772.0,1.0,0.001295,3500,-90
4,content_275b6f7f733016d4,4.866123,810.0,1.0,0.001235,3653,-90


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. The leakage hunt
### The trap: deliberate leakage experiment

Adding a label derived column on purpose to watch a quick score jump toward perfect, then removing it, per the notebook 02 leakage lesson, now on real warehouse data.

In [3]:

# Build a simple proxy label: is this page's CTR "good" (above median)?
import numpy as np
median_ctr = features_df["ctr"].median()
features_df["is_high_ctr"] = (features_df["ctr"] > median_ctr).astype(int)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Honest features only
X_honest = features_df[["avg_position", "impressions_total", "word_count"]].fillna(0)
y = features_df["is_high_ctr"]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(random_state=42).fit(X_train, y_train)
print("Honest accuracy:", model.score(X_test, y_test))

# THE TRAP: add clicks_total, which is literally used to derive ctr/is_high_ctr
X_leaky = features_df[["avg_position", "impressions_total", "word_count", "clicks_total"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = RandomForestClassifier(random_state=42).fit(X_train_l, y_train_l)
print("LEAKY accuracy (clicks_total included):", model_leaky.score(X_test_l, y_test_l))
print("\nclicks_total leaks the answer because ctr is literally derived from it — removing it now.")

Honest accuracy: 0.8566666666666667
LEAKY accuracy (clicks_total included): 1.0

clicks_total leaks the answer because ctr is literally derived from it — removing it now.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
# Trap removed — proceeding only with honest features
print("Keeping the honest number: 0.857 accuracy, using avg_position, impressions_total, word_count only.")
print("clicks_total is excluded going forward — it derives the label itself.")

Keeping the honest number: 0.857 accuracy, using avg_position, impressions_total, word_count only.
clicks_total is excluded going forward — it derives the label itself.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.